# Run 7 - Joint Training with **Equal Per-Domain Training Data**

**Why:** Run6 mixed ~6,370 Kaggle images with only ~900 DeepPCB images (oversampled x4).
Dataset *size* is therefore a confound - we can't tell whether Run6's per-domain scores reflect
modality difficulty or just how many images each domain contributed.

**This experiment removes that confound.** Both domains contribute the **same number of real images**:
Kaggle is randomly down-sampled to match DeepPCB's train count (~900 each, no duplication).
Everything else - model, imgsz, seed, epochs, augmentation, per-domain test sets - is identical to Run6.

Compare the comparison table below against Run6 to see how equalizing the counts shifts each domain's mAP@50.
Training is the only long cell (~30-60 min on an RTX 3090).

## 1. Environment

In [1]:
import torch
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))

PyTorch : 2.5.1+cu124
CUDA    : True
GPU     : NVIDIA GeForce RTX 3090


In [2]:
import subprocess, sys
try:
    import ultralytics, kagglehub, yaml, cv2
except ImportError:
    subprocess.run([sys.executable,"-m","pip","install",
                    "ultralytics","kagglehub","pyyaml","opencv-python-headless","-q"], check=True)
    import ultralytics, kagglehub, yaml, cv2
print("ultralytics:", ultralytics.__version__)
print("opencv     :", cv2.__version__)

ultralytics: 8.4.41
opencv     : 4.10.0


## 2. Configuration

In [3]:
from pathlib import Path
import os

_cwd = Path(os.getcwd()).resolve()
ROOT_ENV = os.environ.get("PCB_PROJECT_ROOT")
if ROOT_ENV:
    PROJECT_ROOT = Path(ROOT_ENV).resolve()
elif _cwd.name == "experiments":
    PROJECT_ROOT = _cwd.parent
elif (_cwd / "experiments").exists():
    PROJECT_ROOT = _cwd
else:
    PROJECT_ROOT = _cwd

SEED, IMG_SIZE, BATCH, DEVICE = 42, 640, 16, 0      # DEVICE='cpu' if no GPU
EPOCHS   = 200                                       # protocol-matched to Run1/Run2/Run6
EXP_NAME = "exp_007_yolov11n_joint_equaldata"

# Unified class space == Run2 training order (DeepPCB conversions already use it)
NAMES = ['mouse_bite','spur','missing_hole','short','open_circuit','spurious_copper']

# DeepPCB raw type (1-6) -> unified index
DEEPPCB_TO_UNIFIED = {1:4, 2:3, 3:0, 4:1, 5:5, 6:2}
# 1=open->open_circuit(4) 2=short(3) 3=mousebite(0) 4=spur(1) 5=copper->spurious(5) 6=pin-hole->missing_hole(2)

DEEPPCB_DIR  = PROJECT_ROOT / "DeepPCB"            # cloned by run3 notebook
DEEPPCB_YOLO = PROJECT_ROOT / "deeppcb_yolo"        # test split already converted by run3
RESULTS_DIR  = PROJECT_ROOT / "results"; RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MERGED = PROJECT_ROOT / "joint_dataset_equal"

print("Project root:", PROJECT_ROOT)
print("DeepPCB repo present  :", DEEPPCB_DIR.exists())
print("DeepPCB test converted:", (DEEPPCB_YOLO/'test'/'images').exists())

Project root: /home/vector/Documents/abdullah_workspace/PCB_Defect_Dectection
DeepPCB repo present  : True
DeepPCB test converted: True


## 3. Locate Kaggle dataset (color domain)

In [4]:
import yaml as _yaml
kaggle_root = kagglehub.dataset_download("norbertelter/pcb-defect-dataset")
KAG_DIR = Path(kaggle_root) / "pcb-defect-dataset"
assert (KAG_DIR/"train"/"images").exists(), f"Kaggle train images missing at {KAG_DIR}"

kag_names_raw = _yaml.safe_load((KAG_DIR/"data.yaml").read_text())["names"]
kag_names = [kag_names_raw[i] for i in range(6)] if isinstance(kag_names_raw,dict) else list(kag_names_raw)
assert kag_names == NAMES, f"Kaggle class order {kag_names} != unified {NAMES}"
print("Kaggle class order matches unified space")
for sp in ["train","valid","val","test"]:
    d = KAG_DIR/sp/"images"
    if d.exists(): print(f"  {sp:6s}: {len(list(d.glob('*')))} images")
KAG_VAL = "valid" if (KAG_DIR/"valid"/"images").exists() else "val"

  train : 8534 images
  val   : 1066 images
  test  : 1068 images


## 4. Convert DeepPCB **trainval** split to YOLO (unified space)

Same as Run6 - reuses the cached conversion if present.

In [5]:
from PIL import Image
import shutil

PCBDATA = DEEPPCB_DIR / "PCBData"
assert PCBDATA.exists(), f"DeepPCB not found at {DEEPPCB_DIR} - run run3 notebook first (it clones the repo)."

tv_list = next(iter(DEEPPCB_DIR.rglob("trainval.txt")), None)
assert tv_list is not None, "trainval.txt not found in DeepPCB."
print("Using:", tv_list.relative_to(DEEPPCB_DIR))

def resolve(rel): return PCBDATA / rel

def convert_split(list_file, out_split):
    img_out = DEEPPCB_YOLO/out_split/"images"; img_out.mkdir(parents=True, exist_ok=True)
    lbl_out = DEEPPCB_YOLO/out_split/"labels"; lbl_out.mkdir(parents=True, exist_ok=True)
    n_img=n_box=skipped=0
    for line in list_file.read_text().splitlines():
        p = line.strip().split()
        if not p: continue
        img_rel = p[0]
        ann_rel = p[1] if len(p)>1 else img_rel.replace("_test.jpg",".txt")
        img_p, ann_p = resolve(img_rel), resolve(ann_rel)
        if not img_p.exists():
            stem = Path(img_rel).stem
            c = list(PCBDATA.rglob(f"{stem}_test.*")) + list(PCBDATA.rglob(f"{stem}_temp.*")) + list(PCBDATA.rglob(Path(img_rel).name))
            img_p = c[0] if c else img_p
        if not ann_p.exists():
            c=list(PCBDATA.rglob(Path(ann_rel).name)); ann_p=c[0] if c else ann_p
        if not img_p.exists() or not ann_p.exists(): skipped+=1; continue
        with Image.open(img_p) as im: W,H = im.size
        rows=[]
        for raw in ann_p.read_text().splitlines():
            v=raw.split()
            if len(v)<5: continue
            x1,y1,x2,y2,t = float(v[0]),float(v[1]),float(v[2]),float(v[3]),int(float(v[4]))
            if t not in DEEPPCB_TO_UNIFIED: continue
            cx,cy,bw,bh = (x1+x2)/2/W,(y1+y2)/2/H,(x2-x1)/W,(y2-y1)/H
            if bw<=0 or bh<=0: continue
            rows.append(f"{DEEPPCB_TO_UNIFIED[t]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"); n_box+=1
        shutil.copy2(img_p, img_out/f"{img_p.stem}.jpg")
        (lbl_out/f"{img_p.stem}.txt").write_text("\n".join(rows))
        n_img+=1
    return n_img, n_box, skipped

if (DEEPPCB_YOLO/"trainval"/"images").exists() and len(list((DEEPPCB_YOLO/"trainval"/"images").glob("*.jpg")))>0:
    print("DeepPCB trainval already converted")
else:
    ni,nb,sk = convert_split(tv_list, "trainval")
    print(f"Converted trainval: {ni} images, {nb} boxes (skipped {sk})")
    assert ni>0 and nb>0

assert (DEEPPCB_YOLO/"test"/"images").exists(), "DeepPCB test split missing - run run3 notebook first."
print("DeepPCB test images:", len(list((DEEPPCB_YOLO/'test'/'images').glob('*.jpg'))))

Using: PCBData/trainval.txt
DeepPCB trainval already converted
DeepPCB test images: 500


## 5. Build the **equal-count** merged training set

- Split DeepPCB trainval 90/10 (seeded) -> ~900 train / ~100 val.
- **Down-sample Kaggle** train to exactly the DeepPCB train count, Kaggle val to the DeepPCB val count.
- Result: 50/50 domain balance with **only real images** (no oversampling/duplication).
- DeepPCB test split stays untouched.

In [6]:
import random as _random, os, shutil

def link_one(ip, lbl_src, img_dst, lbl_dst, suffix=""):
    lp = lbl_src/(ip.stem+".txt")
    if not lp.exists(): return 0
    img_dst.mkdir(parents=True, exist_ok=True); lbl_dst.mkdir(parents=True, exist_ok=True)
    di = img_dst/f"{ip.stem}{suffix}{ip.suffix}"
    dl = lbl_dst/f"{ip.stem}{suffix}.txt"
    if not di.exists():
        try: os.link(ip, di)
        except OSError: shutil.copy2(ip, di)
    if not dl.exists():
        try: os.link(lp, dl)
        except OSError: shutil.copy2(lp, dl)
    return 1

def link_list(img_list, lbl_src, img_dst, lbl_dst, suffix=""):
    n=0
    for ip in img_list: n += link_one(ip, lbl_src, img_dst, lbl_dst, suffix)
    return n

def imgs_in(d):
    return sorted(Path(d).glob("*.[jJpP][pPnN][gG]*"))

# 90/10 split of DeepPCB trainval (seed-matched to Run6 so the val slice is the same)
all_dp = imgs_in(DEEPPCB_YOLO/"trainval"/"images")
_random.seed(SEED); _random.shuffle(all_dp)
cut = max(1, int(len(all_dp) * 0.10))
dp_val_imgs = all_dp[:cut]
dp_tr_imgs  = all_dp[cut:]
N_DP_TR, N_DP_VA = len(dp_tr_imgs), len(dp_val_imgs)
print(f"DeepPCB: {N_DP_TR} train / {N_DP_VA} val")

# Down-sample Kaggle to match DeepPCB counts exactly (equal real images per domain)
# Filter to label-paired images first so sample size == linked count.
kag_tr_all = [ip for ip in imgs_in(KAG_DIR/"train"/"images")
              if (KAG_DIR/"train"/"labels"/(ip.stem+".txt")).exists()]
kag_va_all = [ip for ip in imgs_in(KAG_DIR/KAG_VAL/"images")
              if (KAG_DIR/KAG_VAL/"labels"/(ip.stem+".txt")).exists()]
_random.seed(SEED)
kag_tr_imgs = _random.sample(kag_tr_all, min(N_DP_TR, len(kag_tr_all)))
kag_va_imgs = _random.sample(kag_va_all, min(N_DP_VA, len(kag_va_all)))
print(f"Kaggle down-sampled: {len(kag_tr_imgs)} train / {len(kag_va_imgs)} val")

TR_I, TR_L = MERGED/"train"/"images", MERGED/"train"/"labels"
VA_I, VA_L = MERGED/"val"/"images",   MERGED/"val"/"labels"
if MERGED.exists(): shutil.rmtree(MERGED)   # rebuild cleanly each run

n_kag_tr = link_list(kag_tr_imgs, KAG_DIR/"train"/"labels",   TR_I, TR_L)
n_dp_tr  = link_list(dp_tr_imgs,  DEEPPCB_YOLO/"trainval"/"labels", TR_I, TR_L, suffix="_dp")
n_kag_va = link_list(kag_va_imgs, KAG_DIR/KAG_VAL/"labels",    VA_I, VA_L)
n_dp_va  = link_list(dp_val_imgs, DEEPPCB_YOLO/"trainval"/"labels", VA_I, VA_L, suffix="_dp")

print(f"\ntrain: {n_kag_tr} Kaggle + {n_dp_tr} DeepPCB = {n_kag_tr+n_dp_tr}  (balance {n_kag_tr}:{n_dp_tr})")
print(f"val  : {n_kag_va} Kaggle + {n_dp_va} DeepPCB = {n_kag_va+n_dp_va}")
assert abs(n_kag_tr - n_dp_tr) <= 1, "train domains are not equal-count!"

joint_yaml = MERGED/"joint.yaml"
joint_yaml.write_text(_yaml.dump({
    "path": str(MERGED), "train": "train/images", "val": "val/images",
    "names": {i:n for i,n in enumerate(NAMES)},
}, sort_keys=False))
print("\n", joint_yaml.read_text())

DeepPCB: 900 train / 100 val
Kaggle down-sampled: 900 train / 100 val

train: 676 Kaggle + 900 DeepPCB = 1576  (balance 676:900)
val  : 73 Kaggle + 100 DeepPCB = 173


AssertionError: train domains are not equal-count!

## 6. Train (YOLOv11n, 200 ep, imgsz 640, seed 42 - protocol identical to Run6)

In [ ]:
from ultralytics import YOLO

AUG = dict(   # identical to Run6 so the only changed variable is the dataset composition
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.5,
    degrees=10.0, translate=0.10, scale=0.5, shear=2.0,
    flipud=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.10, copy_paste=0.0,
)

model = YOLO("yolo11n.pt")
results = model.train(
    data=str(joint_yaml), epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH, device=DEVICE,
    seed=SEED, deterministic=True,
    project=str(RESULTS_DIR), name=EXP_NAME, exist_ok=True,
    patience=50, **AUG,
)
BEST = RESULTS_DIR/EXP_NAME/"weights"/"best.pt"
print("\nbest.pt:", BEST, BEST.exists())

## Per-domain test yamls

In [ ]:
EVAL_DIR = MERGED/"eval"; EVAL_DIR.mkdir(parents=True, exist_ok=True)
BEST = RESULTS_DIR/EXP_NAME/"weights"/"best.pt"

KAG_TEST_ROOT = KAG_DIR
kag_test_yaml = EVAL_DIR/"kaggle_test.yaml"
kag_test_yaml.write_text(_yaml.dump({
    "path": str(KAG_TEST_ROOT), "train":"test/images","val":"test/images","test":"test/images",
    "names": {i:n for i,n in enumerate(NAMES)}}, sort_keys=False))

dp_test_yaml = EVAL_DIR/"deeppcb_test.yaml"
dp_test_yaml.write_text(_yaml.dump({
    "path": str(DEEPPCB_YOLO), "train":"test/images","val":"test/images","test":"test/images",
    "names": {i:n for i,n in enumerate(NAMES)}}, sort_keys=False))
print("eval yamls ready")

## Evaluate the joint model on every test domain

In [ ]:
import pandas as pd
from ultralytics import YOLO

joint = YOLO(str(BEST))

def ev(yaml_path, tag):
    r = joint.val(data=str(yaml_path), split="test", imgsz=IMG_SIZE, batch=BATCH, device=DEVICE,
                  project=str(RESULTS_DIR), name=f"{EXP_NAME}_on_{tag}", exist_ok=True,
                  plots=False, verbose=False)
    m50, m = float(r.box.map50), float(r.box.map)
    idx  = list(r.box.ap_class_index)
    ap50 = {int(c):float(a) for c,a in zip(idx, r.box.ap50)}
    pc = pd.DataFrame({"Class":NAMES, "AP@50":[round(ap50.get(i,float('nan')),4) for i in range(6)]})
    print(f"\nJoint -> {tag}:  mAP50={m50:.4f}  mAP50-95={m:.4f}")
    print(pc.to_string(index=False))
    pc.to_csv(RESULTS_DIR/f"{EXP_NAME}_on_{tag}_perclass.csv", index=False)
    return m50, m

j_kag50, j_kag = ev(kag_test_yaml, "kaggle")
j_dp50,  j_dp  = ev(dp_test_yaml,  "deeppcb")

## Compare against Run6 (the unmodified joint baseline)

In [ ]:
import pandas as pd
LABEL = "Run7 equal-data"

# Pull Run6's joint numbers from its saved per-class CSVs (macro-avg AP@50 ~= mAP50)
def run6_map(tag):
    f = RESULTS_DIR/f"exp_006_yolov11n_joint_kaggle_deeppcb_cleanval_on_{tag}_perclass.csv"
    if not f.exists(): return None
    return round(float(pd.read_csv(f)["AP@50"].mean()), 4)

rows = [
    {"Experiment":"Run6 joint (baseline)", "Kaggle test":run6_map("kaggle"), "DeepPCB test":run6_map("deeppcb")},
    {"Experiment":LABEL,                    "Kaggle test":round(j_kag50,4),     "DeepPCB test":round(j_dp50,4)},
]
cmp = pd.DataFrame(rows)
cmp.to_csv(RESULTS_DIR/f"{EXP_NAME}_vs_run6.csv", index=False)
print("====  " + LABEL + " vs Run6  (mAP@50)  ====")
print(cmp.to_string(index=False))
print("\nNote: Run6 per-class CSV means approximate its mAP@50 (macro-avg of the 6 classes).")

## Comparison heatmap

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

cols = ["Kaggle test","DeepPCB test"]
arr = np.array([[r[c] if r[c] is not None else np.nan for c in cols] for _,r in cmp.iterrows()], dtype=float)
fig, ax = plt.subplots(figsize=(7,3.2))
im = ax.imshow(arr, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols)
ax.set_yticks(range(len(cmp))); ax.set_yticklabels(cmp["Experiment"])
for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        v = arr[i,j]
        ax.text(j,i, f"{v:.3f}" if not np.isnan(v) else "N/A", ha="center", va="center",
                fontsize=12, fontweight="bold" if i==1 else "normal",
                color="white" if (not np.isnan(v) and v<0.35) else "black")
plt.colorbar(im, ax=ax)
ax.set_title(f"{EXP_NAME}  vs Run6 (mAP@50)", fontweight="bold")
plt.tight_layout()
hp = RESULTS_DIR/f"{EXP_NAME}_vs_run6_heatmap.png"
plt.savefig(hp, dpi=150, bbox_inches="tight"); plt.show()
print("Saved:", hp)

## Qualitative: joint model on both modalities

In [ ]:
import matplotlib.image as mpimg, random
random.seed(SEED)
kag_imgs = random.sample(imgs_in(KAG_TEST_ROOT/"test"/"images"), 3)
dp_imgs  = random.sample(imgs_in(DEEPPCB_YOLO/"test"/"images"), 3)

fig, axes = plt.subplots(2,3, figsize=(16,9))
for ax, ip in zip(axes.flat, kag_imgs+dp_imgs):
    res = joint.predict(str(ip), imgsz=IMG_SIZE, conf=0.25, verbose=False)[0]
    im = mpimg.imread(str(ip)); ax.imshow(im, cmap="gray")
    for b in res.boxes:
        x1,y1,x2,y2 = b.xyxy[0].tolist()
        ax.add_patch(plt.Rectangle((x1,y1),x2-x1,y2-y1, lw=1.5, edgecolor="red", facecolor="none"))
        ax.text(x1,max(y1-3,0), f"{NAMES[int(b.cls)]} {float(b.conf):.2f}", color="red", fontsize=7)
    ax.set_title(ip.stem, fontsize=8); ax.axis("off")
plt.suptitle("Run7 equal-data joint - top: Kaggle (color) | bottom: DeepPCB (binary)", fontweight="bold")
plt.tight_layout()
qp = RESULTS_DIR/f"{EXP_NAME}_qual_both_domains.png"
plt.savefig(qp, dpi=120); plt.show()
print("Saved:", qp)

## What this isolates

Run6 vs Run7 differ in **one** thing: Run6 gave Kaggle ~7x more images than DeepPCB (with DeepPCB
oversampled x4); Run7 gives both domains the same number of real images.

- If DeepPCB's score **holds up** under equal counts -> the binary modality is genuinely easy and
  Run6's strong DeepPCB number was not just a data-volume artifact.
- If Kaggle's score **drops** sharply -> the color modality needed its larger volume; equalizing
  starves it, and Run6's balance was actually favourable to Kaggle.
- The delta on each domain quantifies the **data-volume confound** in the Run6 joint result.